### Isolation Forest Tests

This notebook illustrates selected remarks made about the isolation forest in Sect. 2.2.1 of the paper, in particular:

1. The correction factor to account for the fact that iTrees are not trained to their maximum granularity.
2. The fact that $\mathbb{E}[h_{j}(\boldsymbol{x})]$  does not necessarily equal $c(\psi)$.

In [9]:
# Imports
import numpy as np
from sklearn.ensemble import IsolationForest as IF
from scipy.stats import norm


def c(n_samples_leaf):
    """
    This function is _average_path_length(n_samples_leaf) taken from https://github.com/scikit-learn/scikit-learn/blob/fe2edb3cd/sklearn/ensemble/_iforest.py#L484.
    One part was commented out.
    """

    # n_samples_leaf = check_array(n_samples_leaf, ensure_2d=False)

    n_samples_leaf_shape = n_samples_leaf.shape
    n_samples_leaf = n_samples_leaf.reshape((1, -1))
    average_path_length = np.zeros(n_samples_leaf.shape)

    mask_1 = n_samples_leaf <= 1
    mask_2 = n_samples_leaf == 2
    not_mask = ~np.logical_or(mask_1, mask_2)

    average_path_length[mask_1] = 0.0
    average_path_length[mask_2] = 1.0
    average_path_length[not_mask] = (
        2.0 * (np.log(n_samples_leaf[not_mask] - 1.0) + np.euler_gamma)
        - 2.0 * (n_samples_leaf[not_mask] - 1.0) / n_samples_leaf[not_mask]
    )

    return average_path_length.reshape(n_samples_leaf_shape)


def z_test(x, mu0):
    """
    Simple function to perform a one-sample z-test for the mean of a sample x against a null hypothesis value mu0. Returns the z-statistic and the two-tailed p-value.
    """

    xbar = np.mean(x)
    se = np.std(x) / np.sqrt(len(x))
    zstat = (xbar - mu0) / se
    pval = 2 * (1 - norm.cdf(np.abs(zstat)))

    return zstat, pval

#### Correction factor test

In [10]:
# Simulate training and test data from a standard normal distribution
rng = np.random.default_rng(seed=42)
Xtrain = rng.normal(loc=0, scale=1, size=[10000, 10])
Xtest = rng.normal(loc=0, scale=1, size=[10000, 10])

# Fit the IF model
if_model = IF(n_estimators=100, max_samples=256, contamination="auto", random_state=42)
if_model.fit(Xtrain)

# compute the training and test scores
test_scores = -if_model.score_samples(Xtest)
train_scores = -if_model.score_samples(Xtrain)

In [11]:
# use information from the trees to compute the path lengths for the training data from scratch, and then compute the corrected path lengths.
path_lengths = np.zeros((len(Xtrain), if_model.n_estimators))
corrected_path_lengths = np.zeros((len(Xtrain), if_model.n_estimators))
for i, tree in enumerate(if_model.estimators_):
    leaves = tree.apply(Xtrain)
    leaf_sizes = tree.tree_.n_node_samples[leaves]
    path_lengths[:, i] = np.ravel(tree.decision_path(Xtrain).sum(axis=1)) - 1
    corrected_path_lengths[:, i] = path_lengths[:, i] + c(leaf_sizes)

# compute the raw and corrected training scores from the path lengths, and compare them to the training scores computed by the model.
h = np.mean(path_lengths, axis=1)
raw_train_scores = 2 ** (-h / c(np.array([256])))
h = np.mean(corrected_path_lengths, axis=1)
corrected_train_scores = 2 ** (-h / c(np.array([256])))
print(
    f"Maximum absolute difference between training scores and raw training scores: {np.max(np.abs(train_scores - raw_train_scores))}"
)
print(
    f"Maximum absolute difference between training scores and corrected training scores: {np.max(np.abs(train_scores - corrected_train_scores))}"
)

Maximum absolute difference between training scores and raw training scores: 0.2189600543873031
Maximum absolute difference between training scores and corrected training scores: 3.885780586188048e-16


In [12]:
# use information from the trees to compute the path lengths for the test data from scratch, and then compute the corrected path lengths.
path_lengths = np.zeros((len(Xtest), if_model.n_estimators))
corrected_path_lengths = np.zeros((len(Xtest), if_model.n_estimators))
for i, tree in enumerate(if_model.estimators_):
    leaves = tree.apply(Xtest)
    leaf_sizes = tree.tree_.n_node_samples[leaves]
    path_lengths[:, i] = np.ravel(tree.decision_path(Xtest).sum(axis=1)) - 1
    corrected_path_lengths[:, i] = path_lengths[:, i] + c(leaf_sizes)

# compute the raw and corrected test scores from the path lengths, and compare them to the test scores computed by the model.
h = np.mean(path_lengths, axis=1)
raw_test_scores = 2 ** (-h / c(np.array([256])))
h = np.mean(corrected_path_lengths, axis=1)
corrected_test_scores = 2 ** (-h / c(np.array([256])))
print(
    f"Maximum absolute difference between test scores and raw test scores: {np.max(np.abs(test_scores - raw_test_scores))}"
)
print(
    f"Maximum absolute difference between test scores and corrected test scores: {np.max(np.abs(test_scores - corrected_test_scores))}"
)

Maximum absolute difference between test scores and raw test scores: 0.21900490160949204
Maximum absolute difference between test scores and corrected test scores: 3.885780586188048e-16


#### Expected path length test

Here we show $\mathbb{E}[h_{j}(\boldsymbol{x})] \neq c(\psi)$, at the very least up to the approximations used for $c(.)$.

In [13]:
# non corrected path lengths
mu0 = c(np.array([256]))[0]
h = np.mean(path_lengths, axis=1)
z_stat, p_value = z_test(h, mu0)
print(f"Z-statistic for raw path lengths: {z_stat:.4f}, p-value: {p_value:.4f}")

h_corrected = np.mean(corrected_path_lengths, axis=1)
z_stat_corrected, p_value_corrected = z_test(h_corrected, mu0)
print(
    f"Z-statistic for corrected path lengths: {z_stat_corrected:.4f}, p-value: {p_value_corrected:.4f}"
)

Z-statistic for raw path lengths: -1103.5558, p-value: 0.0000
Z-statistic for corrected path lengths: 175.6653, p-value: 0.0000


One more example using the uniform distribution instead.

In [14]:
rng = np.random.default_rng(seed=42)
Xtrain = rng.uniform(low=0, high=1, size=[10000, 10])
Xtest = rng.uniform(low=0, high=1, size=[10000, 10])

# Fit the IF model
if_model = IF(n_estimators=100, max_samples=256, contamination="auto", random_state=42)
if_model.fit(Xtrain)

IsolationForest(max_samples=256, random_state=42)

In [15]:
# Training data
path_lengths = np.zeros((len(Xtrain), if_model.n_estimators))
corrected_path_lengths = np.zeros((len(Xtrain), if_model.n_estimators))
for i, tree in enumerate(if_model.estimators_):
    leaves = tree.apply(Xtrain)
    leaf_sizes = tree.tree_.n_node_samples[leaves]
    path_lengths[:, i] = np.ravel(tree.decision_path(Xtrain).sum(axis=1)) - 1
    corrected_path_lengths[:, i] = path_lengths[:, i] + c(leaf_sizes)

# non corrected path lengths
mu0 = c(np.array([256]))[0]
h = np.mean(path_lengths, axis=1)
z_stat, p_value = z_test(h, mu0)
print(f"Z-statistic for raw path lengths: {z_stat:.4f}, p-value: {p_value:.4f}")

h_corrected = np.mean(corrected_path_lengths, axis=1)
z_stat_corrected, p_value_corrected = z_test(h_corrected, mu0)
print(
    f"Z-statistic for corrected path lengths: {z_stat_corrected:.4f}, p-value: {p_value_corrected:.4f}"
)

Z-statistic for raw path lengths: -1300.1982, p-value: 0.0000
Z-statistic for corrected path lengths: -53.7435, p-value: 0.0000


In [16]:
# Testing data
path_lengths = np.zeros((len(Xtest), if_model.n_estimators))
corrected_path_lengths = np.zeros((len(Xtest), if_model.n_estimators))
for i, tree in enumerate(if_model.estimators_):
    leaves = tree.apply(Xtest)
    leaf_sizes = tree.tree_.n_node_samples[leaves]
    path_lengths[:, i] = np.ravel(tree.decision_path(Xtest).sum(axis=1)) - 1
    corrected_path_lengths[:, i] = path_lengths[:, i] + c(leaf_sizes)

# non corrected path lengths
mu0 = c(np.array([256]))[0]
h = np.mean(path_lengths, axis=1)
z_stat, p_value = z_test(h, mu0)
print(f"Z-statistic for raw path lengths: {z_stat:.4f}, p-value: {p_value:.4f}")

h_corrected = np.mean(corrected_path_lengths, axis=1)
z_stat_corrected, p_value_corrected = z_test(h_corrected, mu0)
print(
    f"Z-statistic for corrected path lengths: {z_stat_corrected:.4f}, p-value: {p_value_corrected:.4f}"
)

Z-statistic for raw path lengths: -1308.1556, p-value: 0.0000
Z-statistic for corrected path lengths: -56.4251, p-value: 0.0000
